# AgriSense — Model Training & Mobile Export Pipeline
Trains all three research models (LSTM price, Random Forest production, SARIMA seasonal),
generates every graph needed for the documentation and the app, and exports everything
(models + scalers + graphs + a `mobile_export.json`) as a single downloadable zip for the
KivyMD app.

Upload `AgriSense_Dataset_2021_2025_Cleaned.xlsx` when prompted in the next cell.


In [ ]:

!pip -q install openpyxl xlrd statsmodels scikit-learn tensorflow joblib

import os, json, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)
matplotlib.rcParams['figure.dpi'] = 110
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose

tf.random.set_seed(42)
np.random.seed(42)

# ---- Output folders (mirrors what the mobile app / EC05 doc needs) ----
BASE = "/content/agrisense_export"
DIRS = {
    "models":  f"{BASE}/models",
    "scalers": f"{BASE}/scalers",
    "graphs":  f"{BASE}/graphs",
    "data":    f"{BASE}/data",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print("Folders ready:", DIRS)


In [ ]:

# Upload the cleaned dataset (skip this cell if already in /content)
from google.colab import files
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]
print("Using:", DATA_PATH)


In [ ]:

# ---- Load the three sheets exactly as they exist in the cleaned workbook ----
prices_df = pd.read_excel(DATA_PATH, sheet_name="Weekly Prices")
prod_df   = pd.read_excel(DATA_PATH, sheet_name="Production Data")
weather_df = pd.read_excel(DATA_PATH, sheet_name="Weather Data")

prices_df["Date"] = pd.to_datetime(prices_df["Date"])
weather_df["Date"] = pd.to_datetime(weather_df["Date"])

VEGETABLES = sorted(prices_df["Vegetable"].unique().tolist())
DISTRICTS  = sorted(prices_df["District"].unique().tolist())
print("Vegetables:", VEGETABLES)
print("Districts :", DISTRICTS)
print(prices_df.shape, prod_df.shape, weather_df.shape)
prices_df.head()


## Section 01 — Price Forecasting Model (LSTM)
One LSTM per vegetable (both districts pooled as a feature), 13-week lookback / 4-week forecast, matching the EC05 design (rolling averages, lags, seasonal & festival-style indicators).

In [ ]:

LOOKBACK = 13   # weeks
HORIZON  = 1    # 1-week-ahead forecast (rolled forward for multi-week output at inference)

def build_price_features(df, veg):
    d = df[df["Vegetable"] == veg].copy()
    d = d.sort_values(["District", "Date"]).reset_index(drop=True)
    d["District_enc"] = d["District"].map({dist: i for i, dist in enumerate(DISTRICTS)})
    d["Season_enc"] = d["Season"].map({"Maha": 0, "Yala": 1})
    d["month"] = d["Date"].dt.month
    d["week_sin"] = np.sin(2 * np.pi * d["Week No"] / 52)
    d["week_cos"] = np.cos(2 * np.pi * d["Week No"] / 52)

    out = []
    for dist in DISTRICTS:
        g = d[d["District"] == dist].sort_values("Date").reset_index(drop=True)
        g["ma_4"]  = g["Price (Rs/kg)"].rolling(4, min_periods=1).mean()
        g["ma_12"] = g["Price (Rs/kg)"].rolling(12, min_periods=1).mean()
        g["lag_1"] = g["Price (Rs/kg)"].shift(1)
        g["lag_4"] = g["Price (Rs/kg)"].shift(4)
        g["yoy_change"] = g["Price (Rs/kg)"].pct_change(52)
        g = g.bfill().ffill()
        out.append(g)
    return pd.concat(out).sort_values(["District", "Date"]).reset_index(drop=True)

FEATURE_COLS = ["Price (Rs/kg)", "ma_4", "ma_12", "lag_1", "lag_4",
                 "yoy_change", "District_enc", "Season_enc", "month",
                 "week_sin", "week_cos"]

def make_sequences(arr, lookback):
    X, y = [], []
    for i in range(len(arr) - lookback):
        X.append(arr[i:i + lookback])
        y.append(arr[i + lookback, 0])   # target = Price (Rs/kg), col 0
    return np.array(X), np.array(y)

def build_lstm(n_features):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(LOOKBACK, n_features)),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1, activation="linear"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss="mse")
    return model

price_results = {}

for veg in VEGETABLES:
    feat_df = build_price_features(prices_df, veg)
    values = feat_df[FEATURE_COLS].values.astype(float)

    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values)

    X, y = make_sequences(scaled, LOOKBACK)
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    model = build_lstm(n_features=len(FEATURE_COLS))
    es = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
    history = model.fit(X_train, y_train, validation_split=0.15, epochs=100,
                         batch_size=16, callbacks=[es], verbose=0)

    pred_scaled = model.predict(X_test, verbose=0).flatten()

    def inverse_price(col_vals):
        dummy = np.zeros((len(col_vals), len(FEATURE_COLS)))
        dummy[:, 0] = col_vals
        return scaler.inverse_transform(dummy)[:, 0]

    y_test_actual = inverse_price(y_test)
    y_pred_actual = inverse_price(pred_scaled)

    rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
    mae  = mean_absolute_error(y_test_actual, y_pred_actual)
    mape = np.mean(np.abs((y_test_actual - y_pred_actual) / np.maximum(y_test_actual, 1))) * 100
    dir_actual = np.sign(np.diff(y_test_actual))
    dir_pred   = np.sign(np.diff(y_pred_actual))
    dir_acc = np.mean(dir_actual == dir_pred) * 100 if len(dir_actual) else np.nan

    price_results[veg] = dict(rmse=rmse, mae=mae, mape=mape, dir_acc=dir_acc)
    print(f"{veg:10s}  RMSE={rmse:7.2f}  MAE={mae:7.2f}  MAPE={mape:5.2f}%  DirAcc={dir_acc:5.2f}%")

    # ---- save model + scaler ----
    model.save(f"{DIRS['models']}/price_lstm_{veg.lower()}.keras")
    joblib.dump(scaler, f"{DIRS['scalers']}/price_scaler_{veg.lower()}.pkl")

    # ---- forecast vs actual graph ----
    plt.figure()
    plt.plot(y_test_actual, label="Actual", color="#2E7D32")
    plt.plot(y_pred_actual, label="Forecast", color="#EF6C00", linestyle="--")
    plt.title(f"{veg} — Price Forecast vs Actual (LSTM)")
    plt.xlabel("Week (test period)"); plt.ylabel("Price (Rs/kg)")
    plt.legend(); plt.tight_layout()
    plt.savefig(f"{DIRS['graphs']}/price_forecast_vs_actual_{veg.lower()}.png")
    plt.show(); plt.close()

    # ---- training loss curve ----
    plt.figure()
    plt.plot(history.history["loss"], label="train loss")
    plt.plot(history.history["val_loss"], label="val loss")
    plt.title(f"{veg} — LSTM Training Loss")
    plt.xlabel("Epoch"); plt.ylabel("MSE (scaled)")
    plt.legend(); plt.tight_layout()
    plt.savefig(f"{DIRS['graphs']}/price_loss_curve_{veg.lower()}.png")
    plt.show(); plt.close()

pd.DataFrame(price_results).T


## Historical price trend charts (all vegetables, per district)

In [ ]:

for veg in VEGETABLES:
    plt.figure()
    for dist in DISTRICTS:
        g = prices_df[(prices_df["Vegetable"] == veg) & (prices_df["District"] == dist)].sort_values("Date")
        plt.plot(g["Date"], g["Price (Rs/kg)"], label=dist)
    plt.title(f"{veg} — Historical Weekly Price Trend")
    plt.xlabel("Date"); plt.ylabel("Price (Rs/kg)")
    plt.legend(); plt.tight_layout()
    plt.savefig(f"{DIRS['graphs']}/price_trend_{veg.lower()}.png")
    plt.show(); plt.close()

# Price volatility heatmap (vegetable x month, std of weekly price)
prices_df["month"] = prices_df["Date"].dt.month
pivot = prices_df.pivot_table(index="Vegetable", columns="month",
                               values="Price (Rs/kg)", aggfunc="std")
plt.figure(figsize=(10, 4))
plt.imshow(pivot, aspect="auto", cmap="YlOrRd")
plt.colorbar(label="Price Std Dev (Rs/kg)")
plt.xticks(range(12), range(1, 13)); plt.yticks(range(len(pivot.index)), pivot.index)
plt.title("Price Volatility Heatmap (by month)")
plt.xlabel("Month"); plt.tight_layout()
plt.savefig(f"{DIRS['graphs']}/price_volatility_heatmap.png")
plt.show(); plt.close()


## Section 02 — Production Forecasting Model (Random Forest)

In [ ]:

rf_df = prod_df.copy()
rf_df = rf_df.sort_values(["Vegetable", "District", "Year", "Season"]).reset_index(drop=True)

# lag features: previous season's production for the same veg+district
rf_df["prod_lag_1"] = rf_df.groupby(["Vegetable", "District"])["Production Volume (Mt)"].shift(1)
rf_df["prod_lag_1"] = rf_df["prod_lag_1"].fillna(rf_df["Production Volume (Mt)"].median())

rf_df["Veg_enc"]    = rf_df["Vegetable"].astype("category").cat.codes
rf_df["Dist_enc"]   = rf_df["District"].astype("category").cat.codes
rf_df["Season_enc"] = rf_df["Season"].map({"Maha": 0, "Yala": 1})

FEATURES = ["Veg_enc", "Dist_enc", "Year", "Season_enc",
            "Cultivated Area (ha)", "prod_lag_1"]
TARGET = "Production Volume (Mt)"

X = rf_df[FEATURES]
y = rf_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      random_state=42)

rf_model = RandomForestRegressor(n_estimators=150, max_depth=15,
                                  min_samples_split=5, min_samples_leaf=2,
                                  random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / np.maximum(y_test, 1))) * 100
r2   = r2_score(y_test, y_pred)
bias = np.mean((y_pred - y_test) / np.maximum(y_test, 1)) * 100

print(f"RMSE={rmse:.2f}  MAE={mae:.2f}  MAPE={mape:.2f}%  R2={r2:.4f}  Bias={bias:.2f}%")

joblib.dump(rf_model, f"{DIRS['models']}/production_rf.pkl")
joblib.dump(list(X.columns), f"{DIRS['models']}/production_rf_features.pkl")

# ---- forecast vs actual ----
plt.figure()
plt.scatter(y_test, y_pred, alpha=0.7, color="#1565C0")
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, "--", color="gray")
plt.xlabel("Actual Production (Mt)"); plt.ylabel("Predicted Production (Mt)")
plt.title("Random Forest — Production Forecast vs Actual")
plt.tight_layout()
plt.savefig(f"{DIRS['graphs']}/production_forecast_vs_actual.png")
plt.show(); plt.close()

# ---- feature importance ----
importances = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values()
plt.figure()
importances.plot(kind="barh", color="#2E7D32")
plt.title("Random Forest — Feature Importance")
plt.xlabel("Importance"); plt.tight_layout()
plt.savefig(f"{DIRS['graphs']}/production_feature_importance.png")
plt.show(); plt.close()

# ---- production volume bar chart per vegetable ----
for veg in VEGETABLES:
    g = prod_df[prod_df["Vegetable"] == veg].sort_values(["Year", "Season"])
    labels = g["Year"].astype(str) + " " + g["Season"]
    plt.figure(figsize=(10, 4))
    plt.bar(labels, g["Production Volume (Mt)"], color="#66BB6A")
    plt.xticks(rotation=60); plt.ylabel("Production Volume (Mt)")
    plt.title(f"{veg} — Production Volume by Season")
    plt.tight_layout()
    plt.savefig(f"{DIRS['graphs']}/production_volume_{veg.lower()}.png")
    plt.show(); plt.close()


## Section 03 — Seasonal Trend Analysis (SARIMA)

In [ ]:

sarima_results = {}

for veg in VEGETABLES:
    g = prices_df[prices_df["Vegetable"] == veg].groupby("Date")["Price (Rs/kg)"].mean()
    g = g.asfreq("W").interpolate().bfill().ffill()

    # seasonal decomposition (52-week seasonality)
    decomp = seasonal_decompose(g, model="additive", period=52, extrapolate_trend="freq")
    fig = decomp.plot()
    fig.set_size_inches(10, 7)
    fig.suptitle(f"{veg} — Seasonal Decomposition", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{DIRS['graphs']}/sarima_decomposition_{veg.lower()}.png")
    plt.show(); plt.close()

    # SARIMA fit + short forecast
    try:
        sarima_model = SARIMAX(g, order=(1, 1, 1), seasonal_order=(1, 1, 1, 52),
                                enforce_stationarity=False, enforce_invertibility=False)
        sarima_fit = sarima_model.fit(disp=False)
        forecast = sarima_fit.get_forecast(steps=8)
        fc_mean = forecast.predicted_mean
        fc_ci = forecast.conf_int()

        plt.figure()
        plt.plot(g[-40:], label="History")
        plt.plot(fc_mean, label="Forecast", color="#EF6C00")
        plt.fill_between(fc_ci.index, fc_ci.iloc[:, 0], fc_ci.iloc[:, 1],
                          color="#EF6C00", alpha=0.2)
        plt.title(f"{veg} — SARIMA 8-Week Forecast")
        plt.xlabel("Date"); plt.ylabel("Price (Rs/kg)")
        plt.legend(); plt.tight_layout()
        plt.savefig(f"{DIRS['graphs']}/sarima_forecast_{veg.lower()}.png")
        plt.show(); plt.close()

        joblib.dump(sarima_fit, f"{DIRS['models']}/sarima_{veg.lower()}.pkl")
        sarima_results[veg] = dict(aic=float(sarima_fit.aic),
                                    forecast_next_8w=fc_mean.tolist())
    except Exception as e:
        print(f"SARIMA failed for {veg}: {e}")
        sarima_results[veg] = dict(error=str(e))

sarima_results


## Section 04 — Alert System (Z-Score Anomaly Detection)
Same logic the mobile app's Alert System runs on-device: 90-day (≈13-week) rolling Z-score with Medium/High/Critical thresholds.

In [ ]:

def zscore_alerts(series, window=13):
    roll_mean = series.rolling(window, min_periods=5).mean()
    roll_std  = series.rolling(window, min_periods=5).std()
    z = (series - roll_mean) / roll_std.replace(0, np.nan)
    return z

alert_summary = []
for veg in VEGETABLES:
    g = prices_df[prices_df["Vegetable"] == veg].groupby("Date")["Price (Rs/kg)"].mean().sort_index()
    z = zscore_alerts(g)
    severity = pd.cut(z.abs(), bins=[0, 2.0, 2.5, 3.0, np.inf],
                       labels=["Normal", "Medium", "High", "Critical"], right=False)
    n_alerts = (severity != "Normal").sum()
    alert_summary.append({"vegetable": veg, "alerts_triggered": int(n_alerts)})

    plt.figure()
    plt.plot(g.index, z, color="#6A1B9A")
    plt.axhline(2.0, color="orange", linestyle="--", label="Medium (2.0)")
    plt.axhline(2.5, color="darkorange", linestyle="--", label="High (2.5)")
    plt.axhline(3.0, color="red", linestyle="--", label="Critical (3.0)")
    plt.axhline(-2.0, color="orange", linestyle="--")
    plt.axhline(-2.5, color="darkorange", linestyle="--")
    plt.axhline(-3.0, color="red", linestyle="--")
    plt.title(f"{veg} — Price Z-Score Anomaly Monitor")
    plt.xlabel("Date"); plt.ylabel("Z-Score")
    plt.legend(); plt.tight_layout()
    plt.savefig(f"{DIRS['graphs']}/zscore_alerts_{veg.lower()}.png")
    plt.show(); plt.close()

pd.DataFrame(alert_summary)


## Section 05 — Mobile App Export
Writes `mobile_export.json` (everything the KivyMD app / `prediction` & `recommendation`
tables need: latest metrics, next-week price forecast per vegetable+district, next-season
production forecast) then zips models + scalers + graphs + data for download.

In [ ]:

mobile_export = {
    "generated_at": pd.Timestamp.now().isoformat(),
    "price_model_metrics": price_results,
    "production_model_metrics": {
        "rmse": rmse, "mae": mae, "mape": mape, "r2": r2, "bias_pct": bias
    },
    "sarima_aic": {v: r.get("aic") for v, r in sarima_results.items()},
    "alerts_triggered_per_vegetable": alert_summary,
}

# ---- next-step price forecast per vegetable + district (for 'prediction' table) ----
next_price_forecasts = []
for veg in VEGETABLES:
    feat_df = build_price_features(prices_df, veg)
    scaler = joblib.load(f"{DIRS['scalers']}/price_scaler_{veg.lower()}.pkl")
    model = tf.keras.models.load_model(f"{DIRS['models']}/price_lstm_{veg.lower()}.keras")
    for dist in DISTRICTS:
        g = feat_df[feat_df["District"] == dist].sort_values("Date").tail(LOOKBACK)
        if len(g) < LOOKBACK:
            continue
        vals = g[FEATURE_COLS].values.astype(float)
        scaled = scaler.transform(vals)
        pred_scaled = model.predict(scaled.reshape(1, LOOKBACK, len(FEATURE_COLS)), verbose=0)[0][0]
        dummy = np.zeros((1, len(FEATURE_COLS))); dummy[0, 0] = pred_scaled
        pred_price = float(scaler.inverse_transform(dummy)[0, 0])
        next_price_forecasts.append({
            "vegetable": veg, "district": dist,
            "forecast_date": (g["Date"].max() + pd.Timedelta(weeks=1)).strftime("%Y-%m-%d"),
            "predicted_price_rs_per_kg": round(pred_price, 2)
        })

mobile_export["next_week_price_forecast"] = next_price_forecasts

# ---- next-season production forecast (for 'prediction' table) ----
next_prod_forecasts = []
for veg in VEGETABLES:
    for dist in DISTRICTS:
        g = rf_df[(rf_df["Vegetable"] == veg) & (rf_df["District"] == dist)].sort_values(["Year", "Season"])
        if g.empty:
            continue
        last = g.iloc[-1]
        next_season = "Yala" if last["Season"] == "Maha" else "Maha"
        next_year = last["Year"] + 1 if next_season == "Maha" else last["Year"]
        row = pd.DataFrame([{
            "Veg_enc": last["Veg_enc"], "Dist_enc": last["Dist_enc"],
            "Year": next_year, "Season_enc": 0 if next_season == "Maha" else 1,
            "Cultivated Area (ha)": last["Cultivated Area (ha)"],
            "prod_lag_1": last["Production Volume (Mt)"],
        }])[FEATURES]
        pred = float(rf_model.predict(row)[0])
        next_prod_forecasts.append({
            "vegetable": veg, "district": dist, "season": next_season,
            "year": int(next_year), "predicted_production_mt": round(pred, 2)
        })

mobile_export["next_season_production_forecast"] = next_prod_forecasts

with open(f"{DIRS['data']}/mobile_export.json", "w") as f:
    json.dump(mobile_export, f, indent=2, default=str)

print(json.dumps(mobile_export, indent=2, default=str)[:2000])


In [ ]:

import shutil
zip_path = "/content/agrisense_export"
shutil.make_archive(zip_path, "zip", BASE)
print("Zip ready at:", zip_path + ".zip")

from google.colab import files
files.download(zip_path + ".zip")
